# Toto: Domain-specific модель для observability от Datadog

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/36_toto.ipynb)

## Установка зависимостей

In [ ]:
!pip install -q toto-ts torch pandas numpy matplotlib

## Подготовка данных

Toto оптимизирован для observability-данных: метрик мониторинга инфраструктуры.

In [ ]:
import torch
import numpy as np
import pandas as pd

# Имитируем observability-метрики
np.random.seed(42)

# Параметры
n_variates = 10  # 10 разных метрик (CPU, memory, disk, network и т.д.)
time_steps = 512  # история

# Генерируем данные с характерными для observability паттернами
data = []
for i in range(n_variates):
    # Базовый уровень
    base = 50 + np.random.randn(time_steps).cumsum() * 0.5
    
    # Суточная сезонность (для минутных данных)
    seasonality = 10 * np.sin(np.arange(time_steps) / 60 * 2 * np.pi)
    
    # Редкие спайки (характерно для observability)
    spikes = np.zeros(time_steps)
    spike_indices = np.random.choice(time_steps, size=5, replace=False)
    spikes[spike_indices] = np.random.uniform(50, 100, size=5)
    
    # Шум
    noise = np.random.randn(time_steps) * 3
    
    series = base + seasonality + spikes + noise
    series = np.maximum(series, 0)  # метрики обычно неотрицательные
    data.append(series)

# Формат для Toto: [batch, variates, time_steps]
input_data = torch.tensor(np.array(data)).unsqueeze(0).float()
print(f"Input shape: {input_data.shape}")

## Toto: загрузка и прогнозирование

In [ ]:
from toto import TotoForecaster

# Загрузка модели
model = TotoForecaster.from_pretrained("Datadog/Toto-Open-Base-1.0")
model.eval()

print("Модель Toto загружена!")

In [ ]:
# Генерация прогноза
prediction_length = 96
num_samples = 256  # рекомендуется для получения стабильных квантилей

with torch.no_grad():
    forecasts = model.forecast(
        input_data, 
        prediction_length=prediction_length,
        num_samples=num_samples
    )

print(f"Forecast samples shape: {forecasts.shape}")

# Получение точечного прогноза (медиана)
point_forecast = forecasts.median(dim=0)

# Получение интервалов неопределённости
lower_90 = forecasts.quantile(0.05, dim=0)
upper_90 = forecasts.quantile(0.95, dim=0)

print(f"Point forecast shape: {point_forecast.shape}")

## Визуализация прогнозов

In [ ]:
import matplotlib.pyplot as plt

# Визуализируем несколько метрик
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

metric_names = ['CPU Load', 'Memory Usage', 'Network I/O', 'Disk Latency']

for i, ax in enumerate(axes):
    # История (последние 100 точек)
    history = input_data[0, i, -100:].numpy()
    ax.plot(range(len(history)), history, 'b-', label='История')
    
    # Прогноз
    forecast_start = len(history)
    forecast_idx = range(forecast_start, forecast_start + prediction_length)
    
    # Интервал
    ax.fill_between(
        forecast_idx,
        lower_90[0, i, :].numpy(),
        upper_90[0, i, :].numpy(),
        alpha=0.3, color='red', label='90% интервал'
    )
    
    # Медиана
    ax.plot(forecast_idx, point_forecast[0, i, :].numpy(), 
            'r-', linewidth=2, label='Прогноз')
    
    ax.axvline(x=forecast_start, color='gray', linestyle='--', alpha=0.5)
    ax.set_title(metric_names[i])
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

plt.suptitle('Toto: прогнозы для observability-метрик', fontsize=14)
plt.tight_layout()
plt.show()

## Характеристики observability-данных

In [ ]:
# Демонстрация характеристик observability-данных
characteristics = pd.DataFrame({
    'Характеристика': [
        'Высокая частота',
        'Разреженность (sparsity)',
        'Асимметрия (right skew)',
        'Динамическая кардинальность',
        'Исторические аномалии'
    ],
    'Описание': [
        'Данные собираются каждую секунду/минуту',
        '~12% рядов содержат много нулей',
        '~17% метрик с экстремальными всплесками',
        'Контейнеры создаются/уничтожаются динамически',
        'Инциденты прошлого в данных'
    ],
    'Решение в Toto': [
        'Длинный контекст до 8192 точек',
        'Обучение на реальных observability данных',
        'Robust loss + Student-T mixture',
        'Any-variate attention',
        'Per-variate causal scaling'
    ]
})

print("Характеристики observability-данных и решения Toto:")
print(characteristics.to_string(index=False))

## Сравнение с универсальными моделями

In [ ]:
# Результаты на внутреннем бенчмарке Datadog
benchmark_results = pd.DataFrame({
    'Модель': ['Toto', 'Moirai-Large', 'Moirai-Base', 'Chronos-T5-Mini', 'TimesFM'],
    'sMAPE': [0.672, 0.736, 0.742, 0.788, 1.246],
    'sMdAPE': [0.318, 0.365, 0.370, 0.391, 0.639]
})

print("Результаты на observability-бенчмарке Datadog:")
print(benchmark_results.to_string(index=False))

# Визуализация
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(benchmark_results))
width = 0.35

ax.bar([i - width/2 for i in x], benchmark_results['sMAPE'], width, label='sMAPE')
ax.bar([i + width/2 for i in x], benchmark_results['sMdAPE'], width, label='sMdAPE')

ax.set_ylabel('Ошибка (меньше = лучше)')
ax.set_title('Сравнение моделей на observability-данных')
ax.set_xticks(x)
ax.set_xticklabels(benchmark_results['Модель'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()